## Quality Assessment Framework

The quality assessment framework evaluates the effectiveness of bias correction by consolidating multiple performance metrics into three core components. This comprehensive evaluation is implemented through Python functions utilizing **xarray** and **numpy**, with outputs saved as CF-1.8 compliant NetCDF files.

### 1. Core Quality Components

#### Basic Statistical Quality (weight: 0.35)
This component assesses fundamental bias correction performance using traditional verification metrics:
$$ BasicScore = w_{rb} \cdot RB_{norm} + w_{rmse} \cdot RMSE_{norm} + w_{nse} \cdot NSE $$

- $w_{rb}=0.30$: Weights normalized relative bias.
- $w_{rmse}=0.30$: Weights normalized RMSE, which is scaled using exponential normalization.
- $w_{nse}=0.40$: Weights Nash-Sutcliffe Efficiency.

> **Note:** Pearson Correlation is intentionally excluded here because it is already embedded within NSE. Including both would double-count the linear association component. NSE integrates correlation, bias, and variability into a single measure (Gupta et al., 2009).

Each metric is normalized to a scale of [0,1], ensuring comparability.

#### Distribution Quality Score (weight: 0.35)
Evaluates how well precipitation distributions are preserved:
$$ DQS = w_e \cdot Extreme_{percentiles} + w_g \cdot General_{percentiles} + w_v \cdot Variability + w_k \cdot KS_{pvalue} $$

- $w_e=0.45$: Extreme percentile matching (90th, 95th, 99th percentiles) - most important for GPD/DL.
- $w_g=0.25$: General percentile matching (25th, 50th, 75th percentiles).
- $w_v=0.15$: Variability preservation, measured as the standard deviation ratio.
- $w_k=0.15$: KS p-value - formal distribution test (high p-value = similar distributions = good).

Scores are derived using relative errors, adjusted with tolerance factors for each percentile range.

#### Temporal Quality Score (weight: 0.30)
Assesses the preservation of temporal patterns:
$$ TQS = w_{csi} \cdot CSI + w_e \cdot EventTiming + w_s \cdot SpellMatch $$

- $w_{csi}=0.4$: Weights Critical Success Index - the most balanced categorical metric that simultaneously penalizes both misses and false alarms (Wilks, 2011).
- $w_e=0.3$: Event timing, combining POD and FAR.
- $w_s=0.3$: Dry spell preservation, comparing the longest dry spell length in the reference and test datasets.

> **Note:** Pearson Correlation is intentionally excluded here to avoid double-counting with Basic Statistical Quality (where it is embedded within NSE). CSI is used instead as it provides independent information about categorical event detection accuracy.

### 2. Quality Classification System

#### Categorical Quality
The **categorical_quality** variable provides a classification of quality into four levels:
- **1 (Poor):** CQI < 0.4
- **2 (Fair):** CQI 0.4-0.6
- **3 (Good):** CQI 0.6-0.8
- **4 (Excellent):** CQI >= 0.8

#### Continuous Quality Index
The **continuous_quality** variable is a normalized score ranging from [0,1], combining the three core components:
$$ QI = 0.35 \cdot BasicScore + 0.35 \cdot DistributionScore + 0.30 \cdot TemporalScore $$

### 3. Confidence Assessment

The **confidence_level** variable indicates the reliability of the quality assessment, ranging from [0,1]. It combines:
- **MetricConsistency:** Agreement between different quality indicators (RB, NSE, POD, 1-FAR).
- **DistributionAgreement:** KS p-value used DIRECTLY (high p-value = similar distributions = high confidence).
- **SampleSize** (timeseries only): Proportion of valid data points used.

> **Note:** The KS p-value is used directly (not inverted). A high p-value means no statistical evidence that the distributions differ, indicating successful correction and warranting high confidence.

### 4. Implementation Notes

The framework is implemented with attention to:
- **CF-1.8 Compliant NetCDF Outputs**: Ensuring compatibility with scientific conventions.
- **No Metric Double-Counting**: Each metric appears in only one component.
- **Robust Error Handling**: Addressing missing data and masks effectively.
- **Efficient Processing**: Leveraging xarray operations for performance.

### 5. NetCDF Output Variables

The framework generates the following output variables, with detailed metadata and attributes:

| Variable Name                  | Description                                        | Value Range / Classification |
|--------------------------------|----------------------------------------------------|------------------------------|
| **basic_statistical_quality**  | Aggregated score for statistical quality.          | [0, 1]: Higher is better.    |
| **distribution_quality**       | Evaluates the preservation of precipitation distributions. | [0, 1]: Higher is better.    |
| **temporal_quality**           | Measures temporal pattern preservation.            | [0, 1]: Higher is better.    |
| **continuous_quality**         | Combined overall quality score.                    | [0, 1]: Higher is better.    |
| **categorical_quality**        | Categorical classification of quality.             | 1: Poor, 2: Fair, 3: Good, 4: Excellent. |
| **confidence_level**           | Confidence level for the quality assessment.       | [0, 1]: Higher is better.    |

### 6. References

- WMO (2017), Guidelines on the Calculation of Climate Normals.
- Gupta, H. V. et al. (2009), Decomposition of the mean squared error and NSE performance criteria. J. Hydrology, 377(1-2), 80-91.
- Wilks, D.S. (2011), Statistical Methods in the Atmospheric Sciences, 3rd ed.
- Entekhabi et al. (2010), Performance Metrics for Soil Moisture Retrievals.
- Ebert, E. (2007), Methods for verifying satellite precipitation estimates.

In [ ]:
"""
Quality Assessment for Bias Correction Metrics

This script performs comprehensive quality assessment of bias correction results
using multiple metrics and their combinations. It processes the output from the
bias correction metrics calculation and provides various quality indicators.

Key Features:
------------
1. Core Quality Components
   - Basic Statistical Quality (RB, RMSE, NSE)
   - Distribution Matching (percentiles, extremes, KS test)
   - Temporal Pattern Quality (CSI, event timing, dry spells)

2. Quality Assessment Approaches:
   - Categorical Classification (Excellent/Good/Fair/Poor)
   - Continuous Quality Index (0-1 scale)
   - Component-specific Scores

3. Outputs:
   - Quality Assessment NetCDF (CF-1.8 compliant)
   - Component-wise quality scores
   - Confidence levels

Usage:
------
1. Update input/output directory paths
2. Select month and dekad for assessment
3. Run assessment for each bias correction method (LS, LSEQM, LSEQMDL)
"""
# +++++++++++++++++++++++++++++++++++++++++
# Libraries
# +++++++++++++++++++++++++++++++++++++++++

# Import required libraries
import os
import numpy as np
import xarray as xr
import pandas as pd
import logging
from scipy import stats
from typing import Dict, List, Tuple, Union
import traceback

# Set up logging configuration to print messages to the console in real time
logging.basicConfig(
    level=logging.INFO,  # Set the log level to INFO
    format='%(asctime)s - %(levelname)s - %(message)s',  # Customize log format
    handlers=[logging.StreamHandler()],  # Use StreamHandler to output to console
    force=True  # This forces the use of this config, even if something else has set it up before
)

# +++++++++++++++++++++++++++++++++++++++++
# Configurable Directory
# +++++++++++++++++++++++++++++++++++++++++

# Main directory on Google Drive (modify to your directory structure)
main_dir = f'/content/drive/MyDrive/exercises/gfm1609'

# Define the appropriate input and output directory paths
input_dir = f'{main_dir}/data/bc/input'
output_dir = f'{main_dir}/data/bc/output'
mask_file = f'{main_dir}/data/subset/iso3/idn_subset.nc'  # Mask file

# Method-specific directories
methods = ['ls', 'lseqm', 'lseqmdl']
metrics_paths = {
    method: f'{output_dir}/metrics_{method}'
    for method in methods
}
quality_paths = {
    method: f'{output_dir}/quality_{method}'
    for method in methods
}

# Create output directories
for path in quality_paths.values():
    os.makedirs(path, exist_ok=True)

# +++++++++++++++++++++++++++++++++++++++++
# Configurable Parameters
# +++++++++++++++++++++++++++++++++++++++++

# Quality assessment configuration
QUALITY_THRESHOLDS = {
    'excellent': {
        'relative_bias': 0.1,     # Within +/-10%
        'nse': 0.8,
        'rmse': 2.0,             # mm/day
        'csi': 0.7,
        'pod': 0.8,
        'far': 0.2,
        'stdev_ratio': 0.1,      # Deviation from 1
        'ks_pvalue': 0.05        # Statistical significance
    },
    'good': {
        'relative_bias': 0.25,    # Within +/-25%
        'nse': 0.5,
        'rmse': 5.0,             # mm/day
        'csi': 0.5,
        'pod': 0.6,
        'far': 0.3,
        'stdev_ratio': 0.2,      # Deviation from 1
        'ks_pvalue': 0.01        # Statistical significance
    }
}

# Component weights for overall quality
COMPONENT_WEIGHTS = {
    'basic_stats': 0.35,
    'distribution': 0.35,
    'temporal': 0.30
}

# NetCDF output encoding following CF 1.8 Convention
cf18_quality = {
    'basic_statistical_quality': {'dtype': 'float32', 'zlib': True, '_FillValue': np.nan},
    'distribution_quality': {'dtype': 'float32', 'zlib': True, '_FillValue': np.nan},
    'temporal_quality': {'dtype': 'float32', 'zlib': True, '_FillValue': np.nan},
    'continuous_quality': {'dtype': 'float32', 'zlib': True, '_FillValue': np.nan},
    'categorical_quality': {'dtype': 'int32', 'zlib': True, '_FillValue': -9999},
    'confidence_level': {'dtype': 'float32', 'zlib': True, '_FillValue': np.nan}
}

# Global variable to store user's choice (Overwrite, Skip, or Abort)
user_choice = None

# +++++++++++++++++++++++++++++++++++++++++
# Functions
# +++++++++++++++++++++++++++++++++++++++++

# User decision on existing files
def set_user_decision():
    """
    Prompt user for decision on existing files and store it globally.
    """
    global user_choice
    if user_choice is None:
        decision = input(
            "An output file already exists. Do you want to Overwrite (O), Skip (S), or Abort (A): "
        ).upper()
        while decision not in ['O', 'S', 'A']:
            logging.info("Invalid choice. Please choose again.")
            decision = input(
                "Choose an action - Overwrite (O), Skip (S), Abort (A): "
            ).upper()
        user_choice = decision

# ----
# Time index in the dataset is strictly monotonic
def ensure_strict_monotonic_time(
        ds: xr.Dataset
    ) -> xr.Dataset:
    """
    Ensure the time index in the dataset is strictly monotonic, sorted,
    and that duplicates are removed.
    """
    ds = ds.sortby('time')  # sort by time
    # remove duplicate timestamps
    ds = ds.sel(time=~ds.get_index("time").duplicated())

    # drop any non-monotonic entries
    time_diff = ds['time'].diff('time')
    non_monotonic = time_diff <= pd.Timedelta(0)
    if non_monotonic.any():
        logging.warning("Found non-monotonic time steps => will remove them.")
        ds = ds.sel(time=~non_monotonic)
    return ds

# ----
# Time index in the dataset is strictly monotonic
def reindex_and_align_with_monotonicity(
        reference_ds: xr.Dataset,
        secondary_ds: xr.Dataset
    ) -> xr.Dataset:
    """
    Reindex and align 'secondary_ds' to match 'reference_ds' in time & space,
    ensuring both have strictly monotonic time.
    """
    # ensure strict monotonic time on both
    reference_ds  = ensure_strict_monotonic_time(reference_ds)
    secondary_ds  = ensure_strict_monotonic_time(secondary_ds)

    ref_aligned, sec_aligned = xr.align(reference_ds, secondary_ds, join="inner")

    return ref_aligned, sec_aligned

# ----
# Apply the mask to take out the sea
def apply_land_sea_mask(
        data: xr.DataArray,
        mask_file: str
    ) -> xr.DataArray:
    """
    Apply the land-sea mask to the input data. Keep only land areas.

    Parameters
    ----------
    data : xarray.DataArray
        The data to which the land-sea mask should be applied.
    mask_file : str
        Path to the NetCDF file containing the land-sea mask (1=land, 0=sea).

    Returns
    -------
    masked_data : xarray.DataArray
        Data with the land-sea mask applied (only land).
    """
    mask_ds = xr.open_dataset(mask_file)
    land_sea_mask = mask_ds['land']

    # Interpolate the mask to match data resolution
    land_sea_mask_reindexed = land_sea_mask.interp(
        lat=data.lat, lon=data.lon, method="nearest"
    )

    # Apply the mask
    masked_data = data.where(land_sea_mask_reindexed == 1, drop=True)
    masked_data = masked_data.fillna(np.nan)
    masked_data = masked_data.transpose("time", "lat", "lon")  # Just to ensure order

    mask_ds.close()

    return masked_data

# +++++++++++++++++++++++++++++++++++++++++
# Quality Assessment Functions
# +++++++++++++++++++++++++++++++++++++++++

def calculate_basic_statistical_quality(
        metrics: xr.Dataset,
        weights: dict = None
    ) -> xr.DataArray:
    """
    Calculate basic statistical quality score combining fundamental metrics.

    Each metric is normalized to [0, 1] where 1 = perfect:
    - Relative Bias: 1 - min(|RB|, 1). Penalizes both over- and under-estimation.
    - RMSE: exp(-RMSE / 5). Exponential decay; RMSE of 5 mm/day scores ~0.37.
    - NSE: clipped to [0, 1]. Values < 0 (worse than climatology) score 0.

    Note: Pearson Correlation is intentionally excluded here because it is
    already embedded within NSE (Nash-Sutcliffe Efficiency). Including both
    would double-count the linear association component. NSE is preferred
    because it integrates correlation, bias, and variability into a single
    measure (Gupta et al., 2009).

    Parameters
    ----------
    metrics : xr.Dataset
        Dataset containing at least: relative_bias, rmse, nse.
    weights : dict, optional
        Custom weights for each metric. Keys: 'relative_bias',
        'rmse', 'nse'. Must sum to 1.0.
        Default: RB=0.30, RMSE=0.30, NSE=0.40.

    Returns
    -------
    xr.DataArray
        Basic statistical quality score (0-1)
    """
    if weights is None:
        weights = {
            'relative_bias': 0.30,
            'rmse': 0.30,
            'nse': 0.40,
        }

    # Normalize metrics to 0-1 scale
    rb_score = 1 - np.minimum(np.abs(metrics['relative_bias']), 1)
    rmse_score = np.exp(-metrics['rmse'] / 5)  # Scale RMSE
    nse_score = np.maximum(np.minimum(metrics['nse'], 1), 0)

    # Combine scores
    basic_score = (
        weights['relative_bias'] * rb_score +
        weights['rmse'] * rmse_score +
        weights['nse'] * nse_score
    )

    # Cast to float32
    basic_score = basic_score.astype('float32')

    # Add attributes
    basic_score.attrs.update({
        'long_name': 'Basic Statistical Quality Score',
        'units': 'unitless',
        'valid_range': [0, 1],
        'description': 'Weighted combination of RB, RMSE, NSE scores'
    })

    return basic_score

def calculate_distribution_quality(
        metrics: xr.Dataset,
        weights: dict = None
    ) -> xr.DataArray:
    """
    Calculate distribution quality score from percentile matching, variability,
    and formal distribution testing.

    Evaluates four aspects:
    1. Extreme percentile matching (p90, p95, p99) - most important for GPD/DL.
    2. General percentile matching (p25, p50, p75) - bulk distribution.
    3. Variability preservation (stdev_ratio).
    4. KS p-value - formal distribution test (high p-value = similar = good).

    Parameters
    ----------
    metrics : xr.Dataset
        Dataset containing percentile variables, stdev_ratio, and ks_pvalue.
    weights : dict, optional
        Custom weights. Default: extreme=0.45, general=0.25, var=0.15, ks=0.15.

    Returns
    -------
    xr.DataArray
        Distribution quality score (0-1)
    """
    if weights is None:
        weights = {
            'extreme_percentiles': 0.45,
            'general_percentiles': 0.25,
            'variability': 0.15,
            'ks_test': 0.15,
        }

    # Extreme percentile matching (p90, p95, p99) - most important for GPD/DL
    extreme_pairs = [
        ('p90_ref', 'p90_test', 0.3),
        ('p95_ref', 'p95_test', 0.3),
        ('p99_ref', 'p99_test', 0.4)
    ]
    extreme_score = sum(
        w * (1 - np.minimum(np.abs(metrics[t] - metrics[r]) / (metrics[r] + 0.1), 1))
        for r, t, w in extreme_pairs
    )

    # General percentile matching (p25, p50, p75)
    general_pairs = [
        ('p25_ref', 'p25_test', 0.3),
        ('p50_ref', 'p50_test', 0.4),
        ('p75_ref', 'p75_test', 0.3)
    ]
    general_score = sum(
        w * (1 - np.minimum(np.abs(metrics[t] - metrics[r]) / (metrics[r] + 0.1), 1))
        for r, t, w in general_pairs
    )

    # Variability preservation: ideal stdev_ratio = 1
    var_score = 1 - np.minimum(np.abs(1 - metrics['stdev_ratio']), 1)

    # KS p-value: high p-value = no evidence distributions differ = good
    ks_score = np.maximum(np.minimum(metrics['ks_pvalue'], 1), 0)

    # Combine scores
    dist_score = (
        weights['extreme_percentiles'] * extreme_score +
        weights['general_percentiles'] * general_score +
        weights['variability'] * var_score +
        weights['ks_test'] * ks_score
    )

    # Add attributes
    dist_score.attrs.update({
        'long_name': 'Distribution Quality Score',
        'units': 'unitless',
        'valid_range': [0, 1],
        'description': 'Percentile matching + variability + KS test score'
    })

    return dist_score

def validate_metric_data(metrics, var_name):
    """Validate metric data for numerical issues"""
    data = metrics[var_name].values

    # Check for NaN/Inf
    n_nan = np.isnan(data).sum()
    n_inf = np.isinf(data).sum()

    # Check value ranges
    if not np.isnan(data).all():  # Only if we have some non-NaN values
        data_min = np.nanmin(data)
        data_max = np.nanmax(data)
    else:
        data_min = data_max = np.nan

    logging.info(f"Variable {var_name}:")
    logging.info(f"  Shape: {data.shape}")
    logging.info(f"  Range: [{data_min}, {data_max}]")
    logging.info(f"  NaN count: {n_nan}")
    logging.info(f"  Inf count: {n_inf}")

    return n_nan, n_inf, data_min, data_max

def calculate_temporal_quality(
        metrics: xr.Dataset,
        weights: dict = None
    ) -> xr.DataArray:
    """
    Calculate temporal quality score for pattern and event preservation.

    Evaluates three aspects:
    1. CSI (Critical Success Index): the most balanced categorical metric,
       simultaneously penalizing both misses and false alarms (Wilks, 2011).
    2. Event timing: weighted combination of POD and (1-FAR).
    3. Dry spell preservation: how well dry periods are maintained.

    Note: Pearson Correlation is intentionally excluded here to avoid
    double-counting with Basic Statistical Quality, where it is embedded
    within NSE. CSI is used instead as it provides independent information
    about categorical event detection accuracy.

    Parameters
    ----------
    metrics : xr.Dataset
        Dataset containing: csi, pod, far, dsl_ref, dsl_test.
    weights : dict, optional
        Weights for 'csi', 'event_timing', 'spell_preservation'.
        Must sum to 1.0. Default: 0.4, 0.3, 0.3.

    Returns
    -------
    xr.DataArray
        Temporal quality score (0-1)
    """
    required_vars = ['csi', 'pod', 'far', 'dsl_ref', 'dsl_test']
    logging.info("Validating input metrics:")
    for var in required_vars:
        validate_metric_data(metrics, var)

    if weights is None:
        weights = {
            'csi': 0.4,
            'event_timing': 0.3,
            'spell_preservation': 0.3,
        }

    try:
        # Debug: Print available variables in metrics dataset
        logging.info(f"Available variables in metrics dataset: {list(metrics.data_vars)}")

        # Ensure all required variables exist
        missing_vars = [var for var in required_vars if var not in metrics]
        if missing_vars:
            logging.error(f"Missing required variables for temporal quality: {missing_vars}")
            raise ValueError(f"Missing variables: {missing_vars}")

        # Validate data values
        for var in required_vars:
            data = metrics[var].values
            n_nan = np.isnan(data).sum()
            n_inf = np.isinf(data).sum()
            data_min = np.nanmin(data) if not np.isnan(data).all() else np.nan
            data_max = np.nanmax(data) if not np.isnan(data).all() else np.nan
            logging.info(f"Variable {var}:")
            logging.info(f"  Shape: {metrics[var].shape}")
            logging.info(f"  Range: [{data_min}, {data_max}]")
            logging.info(f"  NaN count: {n_nan}")
            logging.info(f"  Inf count: {n_inf}")

            if n_inf > 0:
                logging.error(f"Found {n_inf} infinite values in {var}")
                raise ValueError(f"Infinite values found in {var}")

        # CSI score: already in [0, 1] where 1 = perfect
        try:
            csi_score = xr.DataArray(
                np.maximum(np.minimum(metrics['csi'].values, 1), 0),
                coords=metrics['csi'].coords,
                dims=metrics['csi'].dims
            )
            logging.info("Successfully calculated CSI score")
            logging.info(f"CSI score range: [{np.nanmin(csi_score)}, {np.nanmax(csi_score)}]")
        except Exception as e:
            logging.error(f"Error calculating CSI score: {str(e)}")
            raise

        # Event timing calculation
        try:
            pod_data = metrics['pod'].values
            far_data = metrics['far'].values
            event_score = xr.DataArray(
                0.6 * pod_data + 0.4 * (1 - far_data),
                coords=metrics['pod'].coords,
                dims=metrics['pod'].dims
            )
            logging.info("Successfully calculated event score")
            logging.info(f"Event score range: [{np.nanmin(event_score)}, {np.nanmax(event_score)}]")
        except Exception as e:
            logging.error(f"Error calculating event score: {str(e)}")
            raise

        # Dry spell preservation calculation
        try:
            dsl_ref_vals = metrics['dsl_ref'].fillna(0).values.astype('float64')
            dsl_test_vals = metrics['dsl_test'].fillna(0).values.astype('float64')

            # Handle timedelta-encoded values from some xarray operations
            if dsl_ref_vals.dtype.kind == 'm':  # timedelta
                ns_per_day = np.float64(8.64e13)
                dsl_ref_vals = dsl_ref_vals.astype('float64') / ns_per_day
                dsl_test_vals = dsl_test_vals.astype('float64') / ns_per_day

            spell_score_data = 1 - np.minimum(
                np.abs(dsl_test_vals - dsl_ref_vals) / (dsl_ref_vals + 1e-6), 1
            )

            spell_score = xr.DataArray(
                spell_score_data,
                coords=metrics['dsl_ref'].coords,
                dims=metrics['dsl_ref'].dims
            )
            logging.info("Successfully calculated spell score")
            logging.info(f"Spell score range: [{np.nanmin(spell_score)}, {np.nanmax(spell_score)}]")
        except Exception as e:
            logging.error(f"Error calculating spell score: {str(e)}")
            raise

        # Combine scores
        try:
            temporal_score = xr.DataArray(
                weights['csi'] * csi_score.values +
                weights['event_timing'] * event_score.values +
                weights['spell_preservation'] * spell_score.values,
                coords=csi_score.coords,
                dims=csi_score.dims
            )
            logging.info("Successfully combined scores")
            logging.info(f"Final temporal score range: [{np.nanmin(temporal_score)}, {np.nanmax(temporal_score)}]")
        except Exception as e:
            logging.error(f"Error combining scores: {str(e)}")
            raise

        # Add attributes
        temporal_score.attrs.update({
            'long_name': 'Temporal Quality Score',
            'units': 'unitless',
            'valid_range': [0, 1],
            'description': 'CSI + event detection + dry spell preservation'
        })

        return temporal_score

    except Exception as e:
        logging.error(f"Error in calculate_temporal_quality: {str(e)}")
        logging.error("Full traceback below:\n" + traceback.format_exc())
        raise  # Re-raise so you see the original exception

def calculate_overall_quality(
        basic_score: xr.DataArray,
        dist_score: xr.DataArray,
        temporal_score: xr.DataArray,
        component_weights: dict = None,
        categorical_thresholds: dict = None
    ) -> Tuple[xr.DataArray, xr.DataArray]:
    """
    Calculate overall quality score and categorical classification.

    Parameters
    ----------
    basic_score : xr.DataArray
        Basic statistical quality score [0, 1].
    dist_score : xr.DataArray
        Distribution quality score [0, 1].
    temporal_score : xr.DataArray
        Temporal quality score [0, 1].
    component_weights : dict, optional
        Weights for 'basic_stats', 'distribution', 'temporal'.
        Default: 0.35, 0.35, 0.30.
    categorical_thresholds : dict, optional
        Thresholds for 'excellent', 'good', 'fair'.
        Default: 0.8, 0.6, 0.4.

    Returns
    -------
    Tuple[xr.DataArray, xr.DataArray]
        Continuous quality index (0-1) and categorical classification (1-4)
    """
    if component_weights is None:
        component_weights = {'basic_stats': 0.35, 'distribution': 0.35, 'temporal': 0.30}
    if categorical_thresholds is None:
        categorical_thresholds = {'excellent': 0.8, 'good': 0.6, 'fair': 0.4}

    # Continuous Quality Index
    continuous_quality = (
        component_weights['basic_stats'] * basic_score +
        component_weights['distribution'] * dist_score +
        component_weights['temporal'] * temporal_score
    ).astype('float32')

    # Categorical classification
    cat = xr.full_like(continuous_quality, 1, dtype='int32')
    cat = xr.where(continuous_quality >= categorical_thresholds['fair'], 2, cat)
    cat = xr.where(continuous_quality >= categorical_thresholds['good'], 3, cat)
    cat = xr.where(continuous_quality >= categorical_thresholds['excellent'], 4, cat)
    cat = xr.where(np.isnan(continuous_quality), -9999, cat).astype('int32')

    # Add attributes
    continuous_quality.attrs.update({
        'long_name': 'Continuous Quality Index',
        'units': 'unitless',
        'valid_range': [0, 1],
        'description': 'Weighted average of basic, distribution, and temporal scores'
    })

    cat.attrs.update({
        'long_name': 'Categorical Quality Classification',
        'units': 'category',
        'flag_values': [1, 2, 3, 4],
        'flag_meanings': 'poor fair good excellent',
        'description': 'Categorized quality levels'
    })

    return continuous_quality, cat

def calculate_confidence(
        metrics: xr.Dataset,
        continuous_quality: xr.DataArray
    ) -> xr.DataArray:
    """
    Estimate confidence in the quality assessment.

    Confidence is derived from:
    - Metric consistency: agreement among normalized metrics (low std = high confidence).
    - Distribution agreement: KS p-value used DIRECTLY (NOT inverted).
      High p-value = distributions are similar = high confidence.
    - Sample size (timeseries only): fraction of non-NaN years.

    Parameters
    ----------
    metrics : xr.Dataset
        Dataset containing bias correction metrics
    continuous_quality : xr.DataArray
        Continuous quality index (used for shape reference)

    Returns
    -------
    xr.DataArray
        Confidence level (0-1)
    """
    try:
        # Metric consistency: stdev across normalized metrics
        # Using RB, NSE, POD, 1-FAR - all normalized to [0, 1] where 1 = good
        arr_rb  = np.maximum(1 - np.abs(metrics['relative_bias']), 0)
        arr_nse = np.maximum(np.minimum(metrics['nse'], 1), 0)
        arr_pod = metrics['pod']
        arr_far = 1 - metrics['far']

        stacked = xr.concat([arr_rb, arr_nse, arr_pod, arr_far], dim='_metric')
        consistency = 1 - stacked.std(dim='_metric', skipna=True)

        # Distribution agreement: KS p-value used DIRECTLY
        # High p-value = distributions are similar = high confidence
        # (Previously inverted as 1-p_value, which was incorrect: it gave
        #  high confidence when distributions were MOST different)
        dist_agreement = np.maximum(np.minimum(metrics['ks_pvalue'], 1), 0)

        if 'time' not in metrics.dims:
            # Single dekad
            confidence = 0.6 * consistency + 0.4 * dist_agreement
        else:
            # Timeseries: include sample size factor
            valid_points = ~np.isnan(metrics['relative_bias'])
            sample_factor = valid_points.sum('time') / len(metrics.time)
            confidence = (
                0.4 * sample_factor +
                0.3 * dist_agreement +
                0.3 * consistency
            )

        # Add CF-Convention style attributes
        confidence.attrs.update({
            'long_name': 'Quality Assessment Confidence',
            'units': 'unitless',
            'valid_range': [0, 1],
            'description': 'Reliability of the quality assessment'
        })

        return confidence

    except Exception as e:
        logging.error(f"Error in calculate_confidence: {str(e)}")
        # Return a default confidence value rather than failing
        default_confidence = xr.full_like(continuous_quality, 0.5)
        default_confidence.attrs.update({
            'long_name': 'Quality Assessment Confidence',
            'units': 'unitless',
            'valid_range': [0, 1],
            'description': 'Default confidence level (errors in calculation)'
        })
        return default_confidence

def save_quality_assessment(
        quality_ds: xr.Dataset,
        out_file: str,
        description: str = "Bias Correction Quality Assessment"
    ) -> None:
    """
    Save quality assessment results to NetCDF file following CF-1.8 convention.

    Parameters
    ----------
    quality_ds : xr.Dataset
        Dataset containing quality assessment results
    out_file : str
        Output file path
    description : str, optional
        Dataset description
    """
    if os.path.exists(out_file):
        logging.info(f"File {out_file} already exists.")
        set_user_decision()
        if user_choice == 'S':
            logging.info(f"Skipping file {out_file}")
            return
        elif user_choice == 'A':
            logging.info("Aborting process.")
            raise SystemExit

    # Add global attributes
    quality_ds.attrs.update({
        'title': description,
        'Conventions': 'CF-1.8',
        'institution': 'The World Bank',
        'source': 'Bias Correction Quality Assessment',
        'references': 'WMO Guidelines for Precipitation Verification',
        'history': f'Created on {pd.Timestamp.now()}',
        'creator_name': 'Benny Istanto',
        'creator_role': 'Climate Geographer',
        'creator_email': 'bistanto@worldbank.org'
    })

    # Add coordinate attributes following CF-1.8
    if 'lat' in quality_ds.coords:
        quality_ds.lat.attrs.update({
            'standard_name': 'latitude',
            'long_name': 'Latitude',
            'units': 'degrees_north',
            'axis': 'Y',
            'valid_range': [-90, 90]
        })
    if 'lon' in quality_ds.coords:
        quality_ds.lon.attrs.update({
            'standard_name': 'longitude',
            'long_name': 'Longitude',
            'units': 'degrees_east',
            'axis': 'X',
            'valid_range': [-180, 180]
        })
    if 'time' in quality_ds.coords:
        # Ensure 'units' and 'calendar' are part of the encoding
        if 'units' in quality_ds.time.attrs:
            quality_ds.time.encoding['units'] = quality_ds.time.attrs.pop('units')
        if 'calendar' in quality_ds.time.attrs:
            quality_ds.time.encoding['calendar'] = quality_ds.time.attrs.pop('calendar')

        # Update 'time' variable's attributes
        quality_ds.time.attrs.update({
            'standard_name': 'time',
            'long_name': 'Time',
            'axis': 'T'
        })

    # Dynamically build the encoding dictionary to include only data variables
    encoding = {var: cf18_quality[var] for var in quality_ds.data_vars if var in cf18_quality}

    try:
        quality_ds.to_netcdf(
            out_file,
            engine='netcdf4',
            encoding=encoding
        )
        logging.info(f"Saved quality assessment => {out_file}")
    except Exception as e:
        logging.error(f"Failed to save {out_file}: {str(e)}")

def main():
    """
    Main function to compute quality assessment for both timeseries and single dekad metrics
    for each reference-test combination. Processes each type independently and generates
    two sets of quality assessment outputs. Continues processing even if one metric type
    is missing.
    """
    # Prompt user for month and dekad
    month_input = input("Enter the month (1-12): ").strip()
    dekad_input = input("Enter the dekad (1, 2, or 3): ").strip()

    try:
        month = int(month_input)
        if not (1 <= month <= 12):
            raise ValueError
    except ValueError:
        raise SystemExit("Invalid month. Please provide a number from 1 to 12.")

    try:
        dekad = int(dekad_input)
        if dekad not in [1, 2, 3]:
            raise ValueError
    except ValueError:
        raise SystemExit("Invalid dekad. Must be 1, 2, or 3.")

    logging.info(f"Processing quality assessment for month={month}, dekad={dekad}")

    # Format strings for file naming
    month_str = f"{month:02d}"
    dekad_str = "01" if dekad == 1 else ("11" if dekad == 2 else "21")

    ref_labels = ['cpc', 'imergl', 'imergf']
    method_labels = ['ls', 'lseqm', 'lseqmdl']

    # Track processing statistics
    processed_counts = {
        'single_dekad': 0,
        'timeseries': 0,
        'failed': 0,
        'skipped': 0
    }

    # Process each combination
    for ref_label in ref_labels:
        for method in method_labels:
            logging.info(f"\nProcessing quality assessment: {ref_label} vs {method}")
            metrics_processed = False

            # Construct the correct test label (adding 'imergl_' prefix)
            test_label = f"imergl_{method}"

            # Process Single Dekad Metrics
            single_metrics_fname = f"idn_cli_metrics_{ref_label}_{test_label}_month{month_str}_dekad{dekad_str}.nc4"
            single_metrics_fpath = os.path.join(metrics_paths[method], single_metrics_fname)

            # Process Timeseries Metrics
            ts_metrics_fname = f"idn_cli_metricsts_{ref_label}_{test_label}_month{month_str}_dekad{dekad_str}.nc4"
            ts_metrics_fpath = os.path.join(metrics_paths[method], ts_metrics_fname)

            logging.info(f"Looking for single dekad metrics at: {single_metrics_fpath}")
            logging.info(f"Looking for timeseries metrics at: {ts_metrics_fpath}")

            # Process Single Dekad Metrics
            if os.path.exists(single_metrics_fpath):
                try:
                    logging.info(f"Processing single dekad metrics: {single_metrics_fname}")
                    metrics_ds = xr.open_dataset(single_metrics_fpath)
                    # Quality Calculations
                    basic_score = calculate_basic_statistical_quality(metrics_ds)
                    dist_score = calculate_distribution_quality(metrics_ds)
                    temporal_score = calculate_temporal_quality(metrics_ds)
                    continuous_quality, categorical_quality = calculate_overall_quality(
                        basic_score, dist_score, temporal_score
                    )
                    confidence = calculate_confidence(metrics_ds, continuous_quality)

                    # Combine into a dataset
                    quality_ds = xr.Dataset({
                        'basic_statistical_quality': basic_score,
                        'distribution_quality': dist_score,
                        'temporal_quality': temporal_score,
                        'continuous_quality': continuous_quality,
                        'categorical_quality': categorical_quality,
                        'confidence_level': confidence
                    })

                    # Save output
                    out_fname = f"idn_cli_quality_{ref_label}_{test_label}_month{month_str}_dekad{dekad_str}.nc4"
                    save_quality_assessment(quality_ds, os.path.join(quality_paths[method], out_fname))
                    metrics_ds.close()
                    processed_counts['single_dekad'] += 1
                    metrics_processed = True
                except Exception as e:
                    logging.error(f"Error processing single dekad metrics for {ref_label} vs {test_label}: {str(e)}")
                    processed_counts['failed'] += 1

            # Process Timeseries Metrics
            if os.path.exists(ts_metrics_fpath):
                try:
                    logging.info(f"Processing timeseries metrics: {ts_metrics_fname}")
                    metrics_ds = xr.open_dataset(ts_metrics_fpath)

                    # Ensure 'time' has 'units' and 'calendar' attributes
                    if 'time' in metrics_ds.coords:
                        if 'units' not in metrics_ds.time.attrs:
                            metrics_ds.time.attrs['units'] = 'days since 2001-01-01'
                        if 'calendar' not in metrics_ds.time.attrs:
                            metrics_ds.time.attrs['calendar'] = 'proleptic_gregorian'

                    # Quality Calculations
                    basic_score = calculate_basic_statistical_quality(metrics_ds)
                    dist_score = calculate_distribution_quality(metrics_ds)
                    temporal_score = calculate_temporal_quality(metrics_ds)
                    continuous_quality, categorical_quality = calculate_overall_quality(
                        basic_score, dist_score, temporal_score
                    )
                    confidence = calculate_confidence(metrics_ds, continuous_quality)

                    # Combine into a dataset
                    quality_ds = xr.Dataset({
                        'basic_statistical_quality': basic_score,
                        'distribution_quality': dist_score,
                        'temporal_quality': temporal_score,
                        'continuous_quality': continuous_quality,
                        'categorical_quality': categorical_quality,
                        'confidence_level': confidence
                    })

                    # Save output
                    out_fname = f"idn_cli_qualityts_{ref_label}_{test_label}_month{month_str}_dekad{dekad_str}.nc4"
                    save_quality_assessment(quality_ds, os.path.join(quality_paths[method], out_fname))
                    metrics_ds.close()
                    processed_counts['timeseries'] += 1
                    metrics_processed = True
                except Exception as e:
                    logging.error(f"Error processing timeseries metrics for {ref_label} vs {test_label}: {str(e)}")
                    processed_counts['failed'] += 1

            if not metrics_processed:
                processed_counts['skipped'] += 1

    # Summary
    logging.info("\nProcessing Summary:")
    logging.info(f"Single Dekad Metrics Processed: {processed_counts['single_dekad']}")
    logging.info(f"Timeseries Metrics Processed: {processed_counts['timeseries']}")
    logging.info(f"Failed Processing Attempts: {processed_counts['failed']}")
    logging.info(f"Combinations Skipped (no metrics found): {processed_counts['skipped']}")
    total_attempts = len(ref_labels) * len(method_labels)
    logging.info(f"Total Combinations Attempted: {total_attempts}")

    if processed_counts['single_dekad'] + processed_counts['timeseries'] > 0:
        logging.info("\nQuality assessment completed successfully for available metrics.")
    else:
        logging.warning("\nNo metrics were successfully processed!")

if __name__ == "__main__":
    main()